# Поглавље 7: Прављење Чет Апликација
## Брзи почетак са OpenAI API

Овај нотебук је прилагођен из [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) који укључује нотебуке који приступају [Azure OpenAI](aoai-assignment.ipynb) услугама.

Python OpenAI API ради и са Azure OpenAI моделима, уз неке модификације. Сазнајте више о разликама овде: [Како пребацити између OpenAI и Azure OpenAI ендпоинта помоћу Питона](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# Преглед  
„Велики језички модели су функције које мапирају текст у текст. За дати улазни низ текста, велики језички модел покушава да предвиди који ће текст у следећем уследити“(1). Овај „блиц“ записник ће корисницима представити основне концепте LLM-а, главне захтеве пакета за почетак са AML-ом, лагани увод у дизајн упита и неколико кратких примера различитих случајева употребе. 


## Садржај  

[Преглед](#overview)  
[Како користити OpenAI сервис](#how-to-use-openai-service)  
[1. Како креирати OpenAI сервис](#1.-creating-your-openai-service)  
[2. Инсталација](#2.-installation)    
[3. Акредитације](#3.-credentials)  

[Примери употребе](#use-cases)    
[1. Сажми текст](#1.-summarize-text)  
[2. Класификуј текст](#2.-classify-text)  
[3. Генериши нова имена производа](#3.-generate-new-product-names)  
[4. Фино подешавање класификатора](#4.fine-tune-a-classifier)  

[Референце](#references)


### Направите свој први упит  
Ова кратка вежба пружа основно упутство за слање упита OpenAI моделу за једноставан задатак „сумирање“.


**Кораци**:  
1. Инсталирајте OpenAI библиотеку у вашем Python окружењу  
2. Учитајте стандардне помоћне библиотеке и подесите ваше типичне OpenAI безбедносне акредитиве за OpenAI сервис који сте креирали  
3. Одаберите модел за ваш задатак  
4. Направите једноставан упит за модел  
5. Пошаљите ваш захтев модел API-ју!


### 1. Инсталирање OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. Увези помоћне библиотеке и иницијализуј креденцијале


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. Проналажење правог модела  
Модели као што су GPT-4o и GPT-4o мини могу да разумеју и генеришу природни језик и доступни су на OpenAI платформи са различитим нивоима снаге и брзине погодним за различите задатке.


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## 4. Дизајн упита  

"Магија великих језичких модела је у томе што, обучавајући се да минимизирају ову грешку предвиђања преко огромне количине текста, модели на крају уче концепте корисне за ова предвиђања. На пример, они уче концепте као што су"(1):

* како се слова сложе
* како граматика функционише
* како се парафразира
* како се одговара на питања
* како се води разговор
* како се пише на многим језицима
* како се кодира
* итд.

#### Како контролисати велики језички модел  
"Од свих улаза у велики језички модел, највише утичући је текстуални упит(1).

Велики језички модели се могу подстаћи да генеришу излаз на неколико начина:

Упутство: Реците моделу шта желите
Довршавање: Наведите модел да доврши почетак онога што желите
Демонстрација: Покажите моделу шта желите, било са:
Неколико примера у упиту
Много стотина или хиљада примера у скупу за фину обуку"



#### Постоје три основна правила за креирање упита:

**Покажи и реци**. Јасно накажите шта желите или кроз упутства, примере, или комбинацију оба. Ако желите да модел рангира листу ставки по азбучном реду или да класификује параграф по сентименту, покажите му да је то оно што желите.

**Обезбедите квалитетне податке**. Ако покушавате да направите класификатор или да натерате модел да прати образац, уверите се да има довољно примера. Обавезно проверите своје примере — модел је обично довољно паметан да уочи основне правописне грешке и да вам одговори, али такође може претпоставити да је то намерно и може утицати на одговор.

**Проверите подешавања.** Подаци температура и top_p контролишу колико је модел детерминистички у генерисању одговора. Ако тражите одговор који има само један тачан одговор, онда желите да их подесите ниже. Ако тражите разноврсније одговоре, онда их можете подесити више. Најчешћа грешка људи у овим подешавањима је да претпоставе да су то контроле "паметности" или "креативности".


Извор: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Пошаљи!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Поновите исти позив, како се резултати упоређују?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Сажми Текст  
#### Изазов  
Сажми текст додавањем 'tl;dr:' на крај текста. Примећујете како модел разуме како да изведе низ задатака без додатних упутстава. Можете експериментисати са описнијим упитима од tl;dr да бисте модификовали понашање модела и прилагодили сажимање које добијате(3).  

Недавни радови су показали значајне помаке у многим НЛП задацима и референтним тачкама тако што су претходно тренирани на великом корпусу текста, а затим дорађивани за конкретан задатак. Иако је архитектонски обично независан од задатка, овај метод ипак захтева сетове података за дораду специфичне за задатак са хиљадама или десетинама хиљада примера. Насупрот томе, људи углавном могу да изведу нови језички задатак само из неколико примера или једноставних упутстава – нешто са чиме данашњи НЛП системи још увек углавном имају проблема. Овде показујемо да значајно повећање скала језичких модела много побољшава перформансе без обзира на задатак, чак и у режиму са малим бројем примера, понекад достигао конкурентност са претходним најбољим приступима дораде. 



Сажето  


# Вежбе за више случајева коришћења  
1. Сажми текст  
2. Класификуј текст  
3. Генериши нове називе производа


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Класификуј Текст  
#### Изазов  
Класификуј ставке у категорије које се дају у време извођења. У следећем примеру, пружамо и категорије и текст за класификацију у промпту(*playground_reference). 

Упит купца: Здраво, један тастер на тастатури мог лаптопа се недавно покварио и требало би ми замену:

Класификована категорија:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Генериши нова имена производа
#### Изазов
Креирај имена производа од примера речи. Овде укључујемо у упиту информације о производу за који ћемо генерисати имена. Такође пружамо сличан пример да бисмо показали образац који желимо да добијемо. Подесили смо и високу вредност температуре да повећамо случајност и иновативније одговоре.

Опис производа: Машина за прављење млечних шејкова за кућну употребу
Почетне речи: брз, здрав, компактни.
Имена производа: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Опис производа: Пар ципела које могу да одговарају свакој величини стопала.
Почетне речи: прилагодљив, одговара, омни-одговарајући.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# Референце  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry портал](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Најбоље праксе за фино подешавање GPT модела за класификацију текста](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# За додатну помоћ  
[OpenAI тим за комерцијализацију](AzureOpenAITeam@microsoft.com) 


# Сарадници
* Луи Ли  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитативним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
